# Building Multi-Turn Tool-Calling Datasets with Data Designer

Generate synthetic training data for agentic Reinforcement-Learning using NVIDIA Data Designer to enhance multi-turn tool calling ability. This example focuses on proactive asking.

## Prerequisites

- **NVIDIA API Key** from [build.nvidia.com](https://build.nvidia.com) to access a remote LLM for generation. To create one, go to [API Keys](https://build.nvidia.com/settings/api-keys), sign in, create a key, and keep it ready for the configuration step below. Alternatively, you may choose to use your own endpoint or deployment.
- **Python 3.11+**
- **Tool definition files** in the `tools/` directory (included in this repo)
- Packages: `data-designer`, `pydantic`, `pandas`

## Objectives

## Architecture Overview

## Context

This tutorial is building on the Workplace Assistant example which is a multi-step tutorial where the data was build with the tool schema seeds. This tutorial will use the existing multi-step single-turn data to expand it into multi-turn data.

<maybe worth it to include tutorial from GTC DLI about generating multi-turn from scratch>

With the existing seed data from Workplace assistant, we know that it results in multi-step results which means that there is a correct order and specific information the agent must have to be able to make the correct tool calls. There is a guarantee that given a quality pass checked ambiguous query, the policy model must pro-activetly ask questions to the user.

## Step 1: Import Dependencies

In [1]:
# Verify dependencies are installed

import importlib

required_packages = ["data_designer", "pydantic", "pandas"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    missing_display = ", ".join(missing_packages)
    raise ImportError(
        "Missing required packages: "
        f"{missing_display}. Install them first with `uv pip install -r requirements.txt`."
    )

print("Dependencies available in the active notebook environment.")

Dependencies available in the active notebook environment.


In [2]:
import json
from typing import List
from pydantic import BaseModel, Field
import pandas as pd
import os

# Data Designer imports
from data_designer.config import (
    ChatCompletionInferenceParams,
    DataDesignerConfigBuilder,
    SamplerColumnConfig, 
    SamplerType,
    CategorySamplerParams,
    ExpressionColumnConfig,
    LLMStructuredColumnConfig,
    LLMTextColumnConfig,
    LocalFileSeedSource,
    ModelConfig,
    SamplingStrategy,
    ModelProvider,
)

from data_designer.interface import DataDesigner

/opt/homebrew/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Load Workplace Assistant Queries

This is the seed data! We are using the output of the workplace assistant SDG which gives us the Gym version of the data. For the seed data, we need to flatten the current hierarchical data. Data Designer needs simple columns it can reference with {{column name}}. 

We will extract user_query, ground_truth, category.

In [3]:
import pandas as pd

DATA_DIR = "./"

data_path = os.path.join(DATA_DIR, "workplace_assistant_train-gpt-oss.jsonl")
print(os.path.exists(data_path), os.path.abspath(data_path))
df = pd.read_json(data_path, lines=True)
user_queries = []

for idx, row in df.iterrows():
    user_queries.append({
        "user_query": row["responses_create_params"]["input"][1]["content"],
        "ground_truth": json.dumps(row["ground_truth"]),
        "category": row["category"],
        "tools_json": json.dumps(row["responses_create_params"]["tools"], indent=2),
        "system_prompt": row["responses_create_params"]["input"][0]["content"],
    })
seed_df = pd.DataFrame(user_queries)

# Saving seed DataFrame as a parquet for Data Designer to load and use in the pipeline
seed_df.to_parquet("workplace_assistant_seeds.parquet", index=False)

True /Users/artij/Projects/Gym/resources_servers/workplace_assistant/notebooks/synthetic-data-generation/multiturn/workplace_assistant_train-gpt-oss.jsonl


**`tools/*.json` Schema Format**

Each JSON file in `tools/` defines one database and its associated tools, emulating tool schemas a company might maintain:

| Field | Description |
|-------|-------------|
| `database` | Database name (e.g. `"calendar"`) |
| `description` | Human-readable description of the database |
| `data_schema` | Database structure — `columns` lists field names, `enums` (if present) defines constrained values. It is not currently consumed by the generation pipeline. |
| `tools` | Array of function definitions: `name`, `description`, `parameters` (JSON Schema), `operation_type` |

Only `description` and `tools` are used by the generation pipeline. `data_schema` is retained in the JSON files as realistic metadata context.

## Step 4: Define Output Schemas

In [4]:
from typing import Literal, Optional

class AmbiguityQuery(BaseModel):
    """Ambiguous query state"""
    ambiguous_message: str= Field(..., description="Natural language rewrite of the original request with ambiguity introduced")
    removed_info: str = Field(..., description="Information removed/hidden from the original query to guide the simulated user")
    clarification_targets: List[str] = Field(..., description= "Questions that the agent can ask to clarify information from the user")
    clarification_requirement: Literal["required", "beneficial", "preference"] = Field(..., description="Either: required, beneficial, preference")

class ToolCall(BaseModel):
    """A single tool invocation."""
    name: str = Field(..., description="The name of the tool to call (e.g., 'email_send_email')")
    arguments: str = Field(..., description="JSON string of the tool arguments")

class ConversationTurn(BaseModel):
    "A single conversation turn between the agent and the user"
    role: Literal["agent", "user"] = Field(..., description="Who is speaking in the conversation")
    content: str = Field(..., description= "Final response from the agent or user")
    thought: Optional[str] = Field(None, description="Reasoning content for the agent only of what to do next and why. Should explain the tool call or question asked")
    tool_calls: Optional[List[ToolCall]] = Field(None, description="Tool calls made (for the agent only). Up to 5-6 tool calls can be made in one turn for the agent.")
    expected_result: Optional[str] = Field(None, description="What information or state change we expect from this.")

class MultiTurnConversation(BaseModel):
    """Full simulated conversation between the user and the agent"""
    conversation_trace: List[ConversationTurn] = Field(..., description="The sequence of conversation turns to solve the task. Should be 2-5 conversation turns.")
    final_answer: str = Field(..., description="A brief confiromation of what was accomplished")

In [5]:
class UserQueryJudgesScores(BaseModel):
    """Quality sctores for the ambiguous generated user query"""
    correct_ambiguity_type: int = Field(                                                                                                                                                       
          ..., ge=1, le=5,                                                                                                                                                                       
          description="Does the ambiguous query match the intended ambiguity type? (1=completely wrong type, 5=perfect match)"                                                                   
      )                                                                                                                                                                                          
    naturalness: int = Field(                                                                                                                                                                  
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Does the ambiguous query sound like a real user? (1=robotic/artificial, 5=very natural)"                                                                                  
    )                                                                                                                                                                                          
    clarification_usefulness: int = Field(                                                                                                                                                     
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Would asking clarifying questions meaningfully improve the outcome? (1=clarification pointless, 5=essential)"                                                             
    )                                                                                                                                                                                          
    schema_consistency: int = Field(                                                                                                                                                           
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Are the ambiguous query, removed info, clarification targets, and clarification requirement all consistent with each other? (1=contradictions, 5=fully consistent)"       
    )                                                                                                                                                                                          
    is_valid: bool = Field(                                                                                                                                                                    
        ...,                                                                                                                                                                                   
        description="True if the ambiguous query should be kept, False if it should be discarded"                                                                                              
    )                                                                                                                                                                                          
    issues: str = Field(                                                                                                                                                                       
        ...,                                                                                                                                                                                   
        description="List any issues found, or 'None' if valid"                                                                                                                                
    ) 


class ConversationJudgeScores(BaseModel):                                                                                                                                                      
    """Quality scores for the generated conversation and tool calls"""                                                                                                                         
    clarification_completeness: int = Field(                                                                                                                                                   
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Did the agent ask about all the important missing information before acting? (1=missed key info, 5=covered everything needed)"                                            
    )   
    # Added for the sake of extending this to 2-5 turns, otherwise the agent will ask all questions in one turn
    clarification_structure: int = Field( 
          ..., ge=1, le=5,
          description="Did the agent ask exactly one clarifying question per turn?"
    )
    user_behavior: int = Field(
        ..., ge=1, le=5,
        description="Did the user only answer what was asked without volunteering extra info?"
    )                                                                                                                                                                                       
    tool_validity: int = Field(                                                                                                                                                                
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Are all tool names valid and arguments schema-compliant? (1=invalid tools/args, 5=all valid)"                                                                             
    )                                                                                                                                                                                                                                                                                                                                                                           
    conversation_coherence: int = Field(                                                                                                                                                       
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Does the conversation flow naturally? Does the user respond consistently with the removed info? (1=incoherent, 5=fully natural)"                                          
    )                                                                                                                                                                                          
    task_completion: int = Field(                                                                                                                                                              
        ..., ge=1, le=5,                                                                                                                                                                       
        description="Does the agent fully solve the original task after clarification? (1=incomplete, 5=fully complete)"                                                                       
    )                                                                                                                                                                                          
    is_valid: bool = Field(                                                                                                                                                                    
        ...,                                                                                                                                                                                   
        description="True if the conversation should be kept, False if it should be discarded"                                                                                                 
    )                                                                                                                                                                                       
    issues: str = Field(
        ...,
        description="List any issues found, or 'None' if valid"                                                                                                                                
    ) 


## Step 5: Define Generation Prompts

In [6]:
AMBIGUOUS_QUERY_GENERATION_PROMPT = """
You are creating training data to teach an AI agent when to ask clarifying questions.

**Your Task** Take the existing original user request below and rewrite it to introduce ambiguity according to the specified ambiguity type. 

**Original User Request:**
{{ user_query }}

**Ambiguity Type to Apply:**
{{ ambiguity_type }}

**Available Tools (and JSON schemas):**
{{ tools_json }}

**Ambiguity Type Definitions:**
- **referential_ambiguity**: Replace explicit names, IDs, or specific references with pronouns and vague terms (e.g., "carlos" → "him", "'Task Update on Develop prototype'" → "that update").
  The agent should be unable to resolve who or what is being referred to and it should require the AI agent to ask for missing information.
- **missing_constraint**: Remove a specific detail that narrows the task to the correct action (e.g., remove "last" or the email subject, so the agent knows it's
about Carlos's email but not which one).
- **user_preference_ambiguity**: Remove style, tone, or execution details while keeping the task fully specified (e.g., remove the exact reply wording but keep who to reply to and which
  email). The agent can do the task but doesn't know the user's preference for how to complete it.

**Guidelines**
1. The ambiguous query must sound natural like a real user who just didn't provide the full details
2. Do NOT add new information. Only remove or obscure what's already in the original
3. The removed information must be recoverable through clarifying questions
4. The ambiguous query must still be about the same task as the original
5. For "removed_info", describe what was hidden so the simulated user can answer agent questions
6. For "clarification_targets", list questions that the agent would naturally ask given ONLY the ambiguous query. The agent does NOT know the original query — it can only ask about things that are obviously missing or ambiguous from its perspective. 
7. For "clarification_requirement", use "required" for reference_ambiguity, "beneficial" for missing_constraint, "preference" for user_preference_ambiguity
8. The query should require 2-5 turns with the user to complete

**Examples by Ambiguity Type**

Original Query: "Reply to carlos's last email about 'Task Update on Develop prototype for report generation' with 'Thanks for the update - I will get back to you tomorrow."

- Referential Ambiguity: "Reply to his last email about that update with 'Thanks for the update - I will get back to you tomorrow.'"
- Missing Constraint: "Reply to carlos's email with 'Thanks for the update - I will get back to you tomorrow.'"
- User Preference Ambiguity: "Reply to carlos's last email about 'Task Update on Develop prototype for report generation' and say I'll follow up tomorrow."

**Output:** Return the complete AmbiguityQuery with all fields.
"""

In [7]:
CONVERSATION_SIMULATION_PROMPT = """
You are simulating a multi-turn conversation between a user and a workplace assistant agent.                                                                                                   
Each participant has LIMITED knowledge — you must respect these boundaries.                                                                                                                    
                                                                                                                                                                                                
---                                                                                                                                                                                            
                                                                                                                                                                                                
## USER PERSPECTIVE (what the user knows)                                                                                                                                                      
The user knows:                                                                                                                                                                                
- Their original intent (but stated it ambiguously)                                                                                                                                            
- The information they left out: {{ ambiguity_query.removed_info }}                                                                                                                            
- The system context: {{ system_prompt }}                                                                                                                                                      
                                                                                                                                                                                                
The user does NOT know:                                                                                                                                                                        
- The available tools or their schemas                                                                                                                                                         
- The ground truth tool calls                                                                                                                                                                  
                                                                                                                                                                                                
User rules:                                                                                                                                                                                    
- Starts the conversation with: "{{ ambiguity_query.ambiguous_message }}"                                                                                                                      
- ONLY answers what the agent asks — does not volunteer extra information                                                                                                                      
- Answers naturally based on the removed info above                                                                                                                                            
- Does NOT use technical tool names or argument names in responses 

---                                                                                                                                                                                            
                                                                                                                                                                                                
## AGENT PERSPECTIVE (what the agent knows)                                                                                                                                                    
The agent knows:                                                                                                                                                                               
- The system context: {{ system_prompt }}                                                                                                                                                      
- The available tools and schemas: {{ tools_json }}                                                                                                                                            
- What information is missing from the request: {{ ambiguity_query.removed_info }}                                                                                                             
                                                                                                                                                                                                
The agent does NOT know:                                                                                                                                                                       
- The user's specific answers until the user provides them                                                                                                                                     
                                                                                                                                                                                                
Agent rules:                                                                                                                                                                                
**CRITICAL**: ONLY ask ONE clarifying question per turn (related sub-parts like first and last name count as one). If a query needs multiple pieces of information from the user, ask for them one at a time.                                                                                     
- Each turn includes a `thought` explaining reasoning                                                                                                                                          
- Only asks about information that is actually missing                                                                                                                                         
- After gathering enough information, makes tool calls to complete the task                                                                                                                    
                                                                                                                                                                                                
---   
## GROUND TRUTH (the correct outcome)                                                                                                                                                          
After clarification, the agent MUST make exactly these tool calls with these argument values:                                                                                                  
{{ ground_truth }}                                                                                                                                                                             
                                                                                                                                                                                                
Work backwards from these tool calls to determine what information the agent needs to gather.                                                                                                  
Do NOT use placeholder variables or placeholder IDs like "00000001" — use the exact values from ground truth.                                                                                  
                                                                                                                                                                                                
---                                                                                                                                                                                            
                                                                                                                                                                                                
## FORMAT                                                                                                                                                                                      
- Agent turns: `role`, `content`, `thought`, and optionally `tool_calls`                                                                                                                       
- User turns: `role` and `content` only (no thought, no tool_calls)                                                                                                                            
- 2-5 turns total                                                                                                                                                                              
- No simulated tool output                                                                                                                                                                     
                                                                                                                                                                                                
**Example:**                                                                                                                                                                                   
Ambiguous Query: "Reply to his last email about that update."                                                                                                                                  
                                                                                                                                                                                                
{% raw %}                                                                                                                                                                                   
```json
[
    {
      "role": "user",
      "content": "Reply to his last email about that update",                                                                                                                                  
      "thought": null,                                                                                                                                                                         
      "tool_calls": null                                                                                                                                                                       
    },                                                                                                                                                                                         
    {                                                                                                                                                                                          
      "role": "agent",                                                                                                                                                                      
      "content": "Sure! Who is the person whose email you'd like me to reply to?",
      "thought": "The user said 'his' but didn't specify who. I need a name to search emails.",                                                                                                
      "tool_calls": null                                                                                                                                                                       
    },                                                                                                                                                                                         
    {                                                                                                                                                                                          
      "role": "user",                                                                                                                                                                          
      "content": "Carlos",                                                                                                                                                                     
      "thought": null,                                                                                                                                                                         
      "tool_calls": null                                                                                                                                                                       
    },                                                                                                                                                                                         
    {                                                                                                                                                                                          
      "role": "agent",                                                                                                                                                                      
      "content": "Got it. Which update are you referring to?",
      "thought": "Now I know it's Carlos, but 'that update' is vague. I need the specific topic.",                                                                                             
      "tool_calls": null                                                                                                                                                                       
    },                                                                                                                                                                                         
    {                                                                                                                                                                                          
      "role": "user",                                                                                                                                                                          
      "content": "The one about the prototype for report generation",                                                                                                                          
      "thought": null,                                                                                                                                                                         
      "tool_calls": null                                                                                                                                                                       
    },                                                                                                                                                                                         
    {                                                                                                                                                                                          
      "role": "agent",                                                                                                                                                                      
      "content": "Done! I've replied to Carlos's email about the task update on the prototype.",
      "thought": "Now I have all the info. Search for the email and reply.",                                                                                                                   
      "tool_calls": [                                                                                                                                                                          
        {"name": "email_search_emails", "arguments": "{\"query\": \"carlos task update prototype report generation\"}"},                                                                       
        {"name": "email_reply_email", "arguments": "{\"email_id\": \"E-005\", \"body\": \"Thanks for the update - I will get back to you tomorrow.\"}"}                                        
      ]                                                                                                                                                                                        
    }                                                                                                                                                                                          
]                                                                                                                                                                                              
{% endraw %}                                                                                                                                                                                   
                                                                                                                                                                                            
Output the complete MultiTurnConversation with all turns needed.
"""

In [8]:
AMBIGUOUS_QUERY_JUDGE_PROMPT = """
You are a quality assurance judge evaluating a synthetically generated ambiguous user query for training an AI workplace assistant to ask clarifying questions.

**Original User Query:**
{{ user_query }}

**Ambiguity Type Requested:**
{{ ambiguity_type }}

**Generated Ambiguous Query:**
{{ ambiguity_query.ambiguous_message }}

**Removed Info:**
{{ ambiguity_query.removed_info }}

**Clarification Targets:**
{{ ambiguity_query.clarification_targets }}

**Clarification Requirement:**
{{ ambiguity_query.clarification_requirement }}

**Available Tools:**
{{ tools_json }}

**Ground Truth Tool Calls:**
{{ ground_truth }}

**Your Task:** Evaluate whether the ambiguous query is well-crafted for training a proactive-asking agent.

**Evaluation Criteria:**

1. **Correct Ambiguity Type (1-5)**: Does the ambiguous query match the requested ambiguity type?
    - Score 1 if the ambiguity is completely the wrong type (e.g., requested referential but only style was removed)
    - Score 3 if it partially matches but also introduces other types of ambiguity
    - Score 5 if it cleanly matches the requested type

2. **Naturalness (1-5)**: Does the ambiguous query sound like a real user?
    - Score 1 if it sounds artificial or forced
    - Score 3 if it's passable but slightly awkward
    - Score 5 if it sounds like something a real person would naturally say

3. **Clarification Usefulness (1-5)**: Would asking clarifying questions meaningfully improve the outcome?
    - Score 1 if the ground truth tool calls can be made without the removed info
    - Score 3 if clarification would help but the agent could make a reasonable guess
    - Score 5 if the agent cannot make the correct ground truth tool calls without the removed information
    - IMPORTANT: Compare the removed info against the ground truth tool calls. If the removed detail does not appear in any tool call argument, score this as 1.

4. **Schema Consistency (1-5)**: Are all the outputs consistent with each other?
    - Score 1 if there are contradictions (e.g., removed_info doesn't match what was actually removed, clarification targets don't address the removed info)
    - Score 3 if mostly consistent with minor gaps
    - Score 5 if ambiguous query, removed info, clarification targets, and clarification requirement are all perfectly aligned

**is_valid:** Set to False if correct_ambiguity_type < 3 OR schema_consistency < 3. These examples should be discarded.

**issues:** List specific problems found. Examples:
- "Requested referential_ambiguity but the query only removed style preferences, not references"
- "removed_info says 'Carlos' was removed but 'Carlos' still appears in the ambiguous query"
- "Clarification targets ask about the email subject but that information is still in the ambiguous query"
- "None" if no issues found

**Output:** Return UserQueryJudgesScores with all fields.
"""

In [9]:
CONVERSATION_JUDGE_PROMPT = """
You are a quality assurance judge evaluating a generated multi-turn conversation for training an AI workplace assistant.

**Original User Query:**
{{ user_query }}

**Ambiguous Query:**
{{ ambiguity_query.ambiguous_message }}

**Removed Info:**
{{ ambiguity_query.removed_info }}

**Clarification Targets:**
{{ ambiguity_query.clarification_targets }}

**Generated Conversation:**
{{ conversation }}

**Available Tools (with full schemas):**
{{ tools_json }}

**Ground Truth Tool Calls:**
{{ ground_truth }}

**Your Task:** Evaluate whether the conversation follows the required structure and correctly solves the task.

**Required Conversation Rules:**
- The user starts with the ambiguous request
- The agent asks ONE clarifying question per turn. Related sub-parts of a single entity (e.g., first and last name) count as one question and can be asked together.
- The user ONLY answers what was asked (does not volunteer extra info)
- The agent makes tool calls only after gathering enough information
- Tool names and arguments must match available tool schemas exactly

**CRITICAL - Check for Argument Validity:**
Tool arguments must use EXACTLY the values allowed by the tool schemas. For example:
- `list_name` must be one of: 'Backlog', 'In Progress', 'In Review', 'Completed'
- `board` must be one of: 'Back end', 'Front end', 'Design'
- `status` must be one of: 'Qualified', 'Won', 'Lost', 'Lead', 'Proposal'
- `product_interest` must be one of: 'Software', 'Hardware', 'Services', 'Consulting', 'Training'

**Evaluation Criteria:**

1. **Clarification Completeness (1-5)**: Did the agent ask about the important missing information before acting?
    - Score 1 if the agent skipped clarification and guessed, or asked about irrelevant things
    - Score 3 if the agent asked about some missing info but missed key details
    - Score 5 if the agent identified and asked about all the critical missing information

2. **Clarification Structure (1-5)**: Did the agent follow the one-question-per-turn rule? 
    **NOTE**: Related sub-parts of a single entity (e.g., first and last name of the same person) count as ONE question, not two.
    - Score 1 if the agent bundled multiple questions in every clarification turn
    - Score 3 if the agent sometimes asked one question and sometimes bundled
    - Score 5 if the agent asked exactly one clarifying question per turn

3. **User Behavior (1-5)**: Did the user follow the rules?
    - Score 1 if the user volunteered information the agent didn't ask for, or dumped all removed info at once
    - Score 3 if the user mostly stayed on topic but occasionally gave extra info
    - Score 5 if the user only answered exactly what the agent asked each turn

4. **Tool Validity (1-5)**: Are all tool names correct and arguments schema-compliant?
    - Score 1 if any tool name doesn't match available tools or arguments use invalid values
    - Score 3 if tools are correct but some arguments are ambiguous
    - Score 5 if all tool names and arguments perfectly match the schemas

5. **Conversation Coherence (1-5)**: Does the conversation flow naturally?
    - Score 1 if the user contradicts the removed info, or the agent ignores user responses
    - Score 3 if mostly natural but with some awkward exchanges
    - Score 5 if both sides respond naturally and consistently throughout

6. **Task Completion (1-5)**: Does the agent fully solve the original task after clarification?
    - Score 1 if the agent never completes the task or makes completely wrong tool calls
    - Score 3 if the agent partially completes the task or misses some ground truth tool calls
    - Score 5 if the agent's tool calls match the ground truth and fully complete the task
    - IMPORTANT: Compare the agent's tool calls against the ground truth tool calls.

**is_valid:** Set to False if clarification_structure < 3 OR user_behavior < 3 OR tool_validity < 4 OR task_completion < 3.

**issues:** List specific problems found. Examples:
- "Agent bundled two unrelated questions in turn 2: asked about both the person and the date"
- "User volunteered the meeting title in turn 3 even though the agent only asked about the date"
- "Agent never asked for clarification and guessed incorrectly"
- "Tool call uses list_name='Todo' but valid values are: 'Backlog', 'In Progress', 'In Review', 'Completed'"
- "Agent's tool calls don't match ground truth: used email_send_email instead of email_reply_email"
- "None" if no issues found

**Output:** Return ConversationJudgeScores with all fields.
"""

## Step 7: Configure the Data Designer Pipeline

In [10]:
import getpass
os.environ["NVIDIA_API_KEY"] = "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"
if "NVIDIA_API_KEY" not in os.environ or not os.environ["NVIDIA_API_KEY"]:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")

In [11]:
# Define custom provider pointing to NVIDIA build.nvidia.com API
NVIDIA_INFERENCE_URL = "https://integrate.api.nvidia.com/v1"

custom_providers = [
    ModelProvider(
        name="nvidia-inference",
        endpoint=NVIDIA_INFERENCE_URL,
        provider_type="openai",
        api_key=os.environ.get("NVIDIA_API_KEY", ""),
    ),
]

# Model name must match NVIDIA's model identifier
USER_MODEL_ID = "openai/gpt-oss-120b" # which model to use at the provider
USER_MODEL_ALIAS = "gpt-oss-120b" # name used in column config

model_configs = [
    ModelConfig(
        alias=USER_MODEL_ALIAS,
        model=USER_MODEL_ID,
        provider="nvidia-inference",
        inference_parameters=ChatCompletionInferenceParams(
            max_tokens=16384,
        ),
    ),
]

# Initialize DataDesigner and config builder
data_designer = DataDesigner(model_providers=custom_providers)
config_builder = DataDesignerConfigBuilder(model_configs=model_configs)

In [12]:
def build_multiturn_pipeline():
    # Initialize the config builder
    config_builder = DataDesignerConfigBuilder(model_configs=model_configs)

    # Load seed data - from Step 2
    config_builder.with_seed_dataset(
        LocalFileSeedSource(path="workplace_assistant_seeds.parquet"),
        sampling_strategy=SamplingStrategy.SHUFFLE
    )

    # Column 1: Ambiguity Type 
    config_builder.add_column(
        SamplerColumnConfig(
            name="ambiguity_type",
            sampler_type=SamplerType.CATEGORY,
            params=CategorySamplerParams(
                values=[
                    '{"type": "referential_ambiguity", "description": "The query should require the AI agent to ask for missing information and perform tool calls once the user provides it in a later turn. This includes replacing explicity references with pronouns and vague terms."}', 
                    '{"type": "missing_constraint", "description": "The query should give the AI agent what task to execute, but remove a specific filter or qualifier needed to narrow it to the correct action. For example, removing which email, which date, or which board."}', 
                    """{"type": "user_preference_ambiguity", "description": "The query should give the AI agent enough information to be able to perform the task, but it does not know the user's preference how to complete the task."}""",
                ],
                weights=[0.3, 0.5, 0.2],
                ),
            )
    )

    # Column 2: Building Ambiguous Query
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="ambiguity_query",
            prompt=AMBIGUOUS_QUERY_GENERATION_PROMPT,
            output_format=AmbiguityQuery,
            model_alias=USER_MODEL_ALIAS,
        )
    )

    # Column 3: Judge the ambiguous query
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="ambiguous_query_judge",
            prompt=AMBIGUOUS_QUERY_JUDGE_PROMPT,
            output_format=UserQueryJudgesScores,
            model_alias=USER_MODEL_ALIAS
        )
    )

    # Column 4: Simulate the conversation
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="conversation",
            prompt=CONVERSATION_SIMULATION_PROMPT,
            output_format=MultiTurnConversation,
            model_alias=USER_MODEL_ALIAS
        )
    )

    # Column 5: Judge the conversation
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="conversation_judge",
            prompt=CONVERSATION_JUDGE_PROMPT,
            output_format=ConversationJudgeScores,
            model_alias=USER_MODEL_ALIAS
        )
    )

    return config_builder

In [13]:
# Build the pipeline
pipeline = build_multiturn_pipeline()
print("Pipeline configured with 4 generation columns:")
print("  1. ambiguous_query (text) - Generate ambiguous user request")
print("  2. ambiguous_query_judge (structured) - Validate query feasibility and schema compliance")
print("  3. conversation (structured) - Generate trajectory after conversation with the user")
print("  4. conversation_judge (structured) - Validate tool calls and argument values")

Pipeline configured with 4 generation columns:
  1. ambiguous_query (text) - Generate ambiguous user request
  2. ambiguous_query_judge (structured) - Validate query feasibility and schema compliance
  3. conversation (structured) - Generate trajectory after conversation with the user
  4. conversation_judge (structured) - Validate tool calls and argument values


In [14]:
data_designer.validate(pipeline)

[09:23:04] [INFO] ✅ Validation passed


In [15]:
preview = data_designer.preview(pipeline, num_records=2)

[09:23:04] [INFO] 🖼️ Preview generation in progress
[09:23:04] [INFO] ✅ Validation passed
[09:23:04] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[09:23:04] [INFO] 🩺 Running health checks for models...
[09:23:04] [INFO]   |-- 👀 Checking 'openai/gpt-oss-120b' in provider named 'nvidia-inference' for model alias 'gpt-oss-120b'...
[09:23:05] [INFO]   |-- ✅ Passed!
[09:23:05] [INFO] 🌱 Sampling 2 records from seed dataset
[09:23:05] [INFO]   |-- seed dataset size: 6 records
[09:23:05] [INFO]   |-- sampling strategy: shuffle
[09:23:05] [INFO] 🎲 Preparing samplers to generate 2 records across 1 columns
[09:23:05] [INFO] (💾 + 💾) Concatenating 2 datasets
[09:23:05] [INFO] 🗂️ llm-structured model config for column 'ambiguity_query'
[09:23:05] [INFO]   |-- model: 'openai/gpt-oss-120b'
[09:23:05] [INFO]   |-- model alias: 'gpt-oss-120b'
[09:23:05] [INFO]   |-- model provider: 'nvidia-inference'
[09:23:05] [INFO]   |-- inference parameters:
[09:23:05] [INFO]   |  |-- generation_ty

In [16]:
preview.dataset

,user_query,ground_truth,category,tools_json,system_prompt,ambiguity_type,ambiguity_query,ambiguous_query_judge,conversation,conversation_judge
0,Please create a follow‑up task for Alex Martin...,"[{""name"": ""company_directory_find_email_addres...",workplace_assistant_customer_relationship_manager,"[\n {\n ""type"": ""function"",\n ""name"": ""...","Today's date is Thursday, 2026-01-29 and the c...","{""type"": ""missing_constraint"", ""description"": ...",{'ambiguous_message': 'Please create a follow‑...,"{'correct_ambiguity_type': 5, 'naturalness': 5...","{'conversation_trace': [{'role': 'user', 'cont...","{'clarification_completeness': 2, 'clarificati..."
1,Could you pull the website stats for last mont...,"[{""name"": ""analytics_total_visits_count"", ""arg...",workplace_assistant_analytics,"[\n {\n ""type"": ""function"",\n ""name"": ""...","Today's date is Thursday, 2026-01-29 and the c...","{""type"": ""missing_constraint"", ""description"": ...",{'ambiguous_message': 'Could you pull the webs...,"{'correct_ambiguity_type': 5, 'naturalness': 5...","{'conversation_trace': [{'role': 'user', 'cont...","{'clarification_completeness': 5, 'clarificati..."


In [17]:
preview.display_sample_record()

                                                 Seed Columns                                                 
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name          ┃ Value                                                                                      ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user_query    │ Please create a follow‑up task for Alex Martinez to reach out to the new prospect Global   │
│               │ Tech Corp about Consulting services, with a follow‑up date of 2024‑05‑20 and set the       │
│               │ status to Lead.                                                                            │
├───────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│ ground_truth  │ [                                                                                          │
│               │     {                                                                                      │
│               │         'name': 'company_directory_find_email_address',                                    │
│               │         'arguments': '{"name": "Alex Martinez"}'                                           │
│               │     },                                                                                     │
│               │     {                                                                                      │
│               │         'name': 'customer_relationship_manager_add_customer',                              │
│               │         'arguments': '{"customer_name": "Global Tech Corp", "assigned_to_email":           │
│               │ "<email_from_step_1>", "status": "Lead", "product_interest": "Consulting", "follow_up_by": │
│               │ "2024-05-20", "notes": "Reach out about Consulting services"}'                             │
│               │     }                                                                                      │
│               │ ]                                                                                          │
├───────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│ category      │ workplace_assistant_customer_relationship_manager                                          │
├───────────────┼────────────────────────────────────────────────────────────────────────────────────────────┤
│ tools_json    │ [                                                                                          │
│               │     {                                                                                      │
│               │         'type': 'function',                                                                │
│               │         'name': 'company_directory_find_email_address',                                    │
│               │         'description': 'Finds all email addresses containing the given name                │
│               │ (case-insensitive search).',                                                               │
│               │         'parameters': {                                                                    │
│               │             'type': 'object',                                                              │
│               │             'properties': {                                                                │
│               │                 'name': {                                                                  │
│               │                     'type': 'string',                                                      │
│               │                     'description': 'Name or partial name to search for in email addresses' │
│               │                 }                                                                          │
│   

## Step 8: Set Up Quality Filtering

In [18]:
from quality_filtering import filter_high_quality, show_rejection_reasons

## Step 9: Generate and Filter the Dataset

Run the full pipeline end-to-end: generate records and apply two levels of **quality filtering**.

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│   Generate   │───▶│ Stage 1:     │───▶│ Stage 2:     │───▶│   Filtered   │
│   Records    │    │ Query Judge  │    │ Convo Judge  │    │   Dataset    │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘

**Utility location:** `utils/quality_filtering.py`

**Quick usage:**
- Run `show_rejection_reasons(results_df, num_examples=3)` to inspect failures
- Run `filter_high_quality(results_df, verbose=True)` to apply default strict filtering
- Optionally tune thresholds with `FilterThresholds(...).to_kwargs()`

**Why Two Levels of Quality Filtering?**
- **Stage 1 (User Query)**: Catches queries which are intractable in this context. Filters on the `UserQueryJudgeScores` fields (`feasibility`, `schema_compliance`, `naturalness`) produced by the LLM judge prompts defined in Step 4.
- **Stage 2 (Trajectory)**: Catches tool argument errors that slipped through, or that doesn't answer the query. Filters on the `TrajectoryJudgeScores` fields (`tool_validity`, `argument_validity`, `completeness`, `efficiency`) from Step 4.


In [19]:
print("Generating 10 examples...")
results = data_designer.create(pipeline, num_records=10)

results_df = results.load_dataset()
print(f"\nGenerated {len(results_df)} records")
print("\nColumns:", list(results_df.columns))

[09:23:33] [INFO] 🎨 Creating Data Designer dataset
[09:23:33] [INFO] 📂 Dataset path '/Users/artij/Projects/Gym/resources_servers/workplace_assistant/notebooks/synthetic-data-generation/multiturn/artifacts/dataset' already exists. Dataset from this session
		     will be saved to '/Users/artij/Projects/Gym/resources_servers/workplace_assistant/notebooks/synthetic-data-generation/multiturn/artifacts/dataset_04-07-2026_092333' instead.
[09:23:33] [INFO] ✅ Validation passed
[09:23:33] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[09:23:33] [INFO] 🩺 Running health checks for models...
[09:23:33] [INFO]   |-- 👀 Checking 'openai/gpt-oss-120b' in provider named 'nvidia-inference' for model alias 'gpt-oss-120b'...


Generating 10 examples...


[09:23:34] [INFO]   |-- ✅ Passed!
[09:23:34] [INFO] ⏳ Processing batch 1 of 1
[09:23:34] [INFO] 🌱 Sampling 10 records from seed dataset
[09:23:34] [INFO]   |-- seed dataset size: 6 records
[09:23:34] [INFO]   |-- sampling strategy: shuffle
[09:23:34] [INFO] 🎲 Preparing samplers to generate 10 records across 1 columns
[09:23:34] [INFO] (💾 + 💾) Concatenating 2 datasets
[09:23:34] [INFO] 🗂️ llm-structured model config for column 'ambiguity_query'
[09:23:34] [INFO]   |-- model: 'openai/gpt-oss-120b'
[09:23:34] [INFO]   |-- model alias: 'gpt-oss-120b'
[09:23:34] [INFO]   |-- model provider: 'nvidia-inference'
[09:23:34] [INFO]   |-- inference parameters:
[09:23:34] [INFO]   |  |-- generation_type=chat-completion
[09:23:34] [INFO]   |  |-- max_parallel_requests=4
[09:23:34] [INFO]   |  |-- max_tokens=16384
[09:23:34] [INFO] ⚡️ Processing llm-structured column 'ambiguity_query' with 4 concurrent workers
[09:23:34] [INFO] ⏱️ llm-structured column 'ambiguity_query' will report progress after ea


Generated 10 records

Columns: ['user_query', 'ground_truth', 'category', 'tools_json', 'system_prompt', 'ambiguity_type', 'ambiguity_query', 'ambiguous_query_judge', 'conversation', 'conversation_judge']


In [20]:
# Show rows the LLM judge already marked is_valid=False before stricter threshold filtering below.
show_rejection_reasons(results_df, num_examples=10)


=== Ambiguous Query Issues (0/10 rejected) ===
  No issues found.

=== Conversation Issues (2/10 rejected) ===
  [1] {'conversation_trace': [{'content': 'Please create a follow‑up task for Alex Martinez to contact the...
      Issues: - Agent skipped all required clarification questions (list, board, task name) and proceeded directly to tool calls.
- Agent did not follow the one‑question‑per‑turn rule (no clarifying turns at all).
- Agent's final answer claims a task was created, but the tool calls only added a customer record, which contradicts the user's request.
  [2] {'conversation_trace': [{'content': 'I want to see how the site performed in March. Please give me t...
      Issues: Agent bundled two unrelated questions (year and chart type) in a single clarification turn, violating the one-question-per-turn rule.


In [21]:
filtered_df = filter_high_quality(
    results_df,
    min_correct_ambiguity_type=3,
    min_naturalness=3,
    min_clarification_usefulness=3,
    min_schema_consistency=4,
    min_clarification_completeness=3,
    min_clarification_structure=3,
    min_user_behavior=3,
    min_tool_validity=4,
    min_conversation_coherence=3,
    min_task_completion=3,
    verbose=True,
)

filtered_df.head()


=== Quality Filtering Results ===
Total records: 10

Stage 1 (Ambiguous Query): 5/10 passed (50%)
  is_valid: 10 | ambiguity_type>=3: 10 | naturalness>=3: 10 | usefulness>=3: 6 | consistency>=4: 9

Stage 2 (Conversation): 8/10 passed (80%)
  is_valid: 8 | clarification>=3: 9 | structure>=3: 8 | user_behavior>=3: 10 | tool_validity>=4: 10 | coherence>=3: 9 | completion>=3: 10

Final: 4/10 passed (40%)


,user_query,ground_truth,category,tools_json,system_prompt,ambiguity_type,ambiguity_query,ambiguous_query_judge,conversation,conversation_judge
0,I need to see how our site performed in March....,"[{""name"": ""analytics_total_visits_count"", ""arg...",workplace_assistant_analytics,"[ { ""type"": ""function"", ""name"": ""com...","Today's date is Thursday, 2026-01-29 and the c...","{""type"": ""missing_constraint"", ""description"": ...",{'ambiguous_message': 'I need to see how our s...,"{'clarification_usefulness': 5, 'correct_ambig...",{'conversation_trace': array([{'content': 'I n...,"{'clarification_completeness': 5, 'clarificati..."
1,Could you pull the website stats for last mont...,"[{""name"": ""analytics_total_visits_count"", ""arg...",workplace_assistant_analytics,"[ { ""type"": ""function"", ""name"": ""com...","Today's date is Thursday, 2026-01-29 and the c...","{""type"": ""referential_ambiguity"", ""description...",{'ambiguous_message': 'Could you pull the webs...,"{'clarification_usefulness': 5, 'correct_ambig...",{'conversation_trace': array([{'content': 'Cou...,"{'clarification_completeness': 3, 'clarificati..."
2,Can you find Maya Patel’s email address and al...,"[{""name"": ""company_directory_find_email_addres...",workplace_assistant_analytics,"[ { ""type"": ""function"", ""name"": ""com...","Today's date is Thursday, 2026-01-29 and the c...","{""type"": ""missing_constraint"", ""description"": ...",{'ambiguous_message': 'Can you find Maya Patel...,"{'clarification_usefulness': 5, 'correct_ambig...",{'conversation_trace': array([{'content': 'Can...,"{'clarification_completeness': 5, 'clarificati..."
3,Can you find Maya Patel’s email address and al...,"[{""name"": ""company_directory_find_email_addres...",workplace_assistant_analytics,"[ { ""type"": ""function"", ""name"": ""com...","Today's date is Thursday, 2026-01-29 and the c...","{""type"": ""user_preference_ambiguity"", ""descrip...",{'ambiguous_message': 'Can you find Maya Patel...,"{'clarification_usefulness': 5, 'correct_ambig...",{'conversation_trace': array([{'content': 'Can...,"{'clarification_completeness': 3, 'clarificati..."


## Step 10: Convert to NeMo Gym Format and Save

In [22]:
import importlib
import convert_to_nemo_gym_format
importlib.reload(convert_to_nemo_gym_format)
from convert_to_nemo_gym_format import convert_to_nemo_gym_format, save_for_nemo_gym

In [23]:
from convert_to_nemo_gym_format import convert_to_nemo_gym_format, save_for_nemo_gym

output_path = "workplace_assistant_multiturn_train.jsonl"
save_for_nemo_gym(filtered_df, output_path, convert_fn=convert_to_nemo_gym_format)

# Display a sample of the saved task data
with open(output_path) as f:
    first_row = json.loads(f.readline())

print("\nSample saved row:")
print(json.dumps(first_row, indent=2))

Saved 4 examples to workplace_assistant_multiturn_train.jsonl

Sample saved row:
{
  "responses_create_params": {
    "input": [
      {
        "role": "system",
        "content": "Today's date is Thursday, 2026-01-29 and the current time is 23:59:00. Remember the current date and time when answering queries. Meetings must not start before 9am or end after 6pm."
      },
      {
        "role": "user",
        "content": "I need to see how our site performed in March. Could you give me the total visits and engaged users for that month, and also generate a line chart showing daily total visits?"
      },
      {
        "role": "assistant",
        "content": "Sure! Which year\u2019s March are you interested in?"
      },
      {
        "role": "user",
        "content": "2023"
      },
      {
        "role": "assistant",
        "content": "Got it. I\u2019ve retrieved the total visits and engaged users for March\u202f2023 and created a line chart of daily total visits.",
        "t

## Summary

## Next Steps